# MNIST Classification with CNN (LeNet-5)


In [ ]:
import copy
import torch
import torch.nn as nn
from torchvision import datasets
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 1. Load MNIST

In [ ]:
train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True
)

# Randomly split the training dataset into training and validation sets
torch.manual_seed(0)
indices = torch.randperm(len(train_dataset))

train_indices = indices[:20000]
val_indices = indices[20000:25000]

X_train = train_dataset.data[train_indices].float() / 255.0
y_train = train_dataset.targets[train_indices]

X_val = train_dataset.data[val_indices].float() / 255.0
y_val = train_dataset.targets[val_indices]

# Use the official MNIST test set only for final evaluation
X_test = test_dataset.data.float() / 255.0
y_test = test_dataset.targets

# LeNet-5 uses 32 x 32 input images.
# MNIST images are 28 x 28, so add 2-pixel zero padding on each side.
X_train = torch.nn.functional.pad(X_train, (2, 2, 2, 2))
X_val = torch.nn.functional.pad(X_val, (2, 2, 2, 2))
X_test = torch.nn.functional.pad(X_test, (2, 2, 2, 2))

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:  ", X_val.shape)
print("y_val:  ", y_val.shape)
print("X_test: ", X_test.shape)
print("y_test: ", y_test.shape)


## 2. Define LeNet-5


In [ ]:
class MyLeNet(nn.Module):
    def __init__(self):
        super().__init__()

        # Convolutional layers
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5, stride=1, padding=0)
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5, stride=1, padding=0)
        self.conv3 = nn.Conv2d(in_channels=16, out_channels=120, kernel_size=5, stride=1, padding=0)

        # Average pooling layer
        self.pool = nn.AvgPool2d(kernel_size=2, stride=2)

        # Fully-connected (fc) layers
        self.fc1 = nn.Linear(120, 84)
        self.fc2 = nn.Linear(84, 10)

        # ReLU activation function
        self.relu = nn.ReLU()

    def forward(self, x):
        # [batch, 1, 32, 32] -> [batch, 6, 28, 28]
        x = self.conv1(x)
        x = self.relu(x)

        # [batch, 6, 28, 28] -> [batch, 6, 14, 14]
        x = self.pool(x)

        # [batch, 6, 14, 14] -> [batch, 16, 10, 10]
        x = self.conv2(x)
        x = self.relu(x)

        # [batch, 16, 10, 10] -> [batch, 16, 5, 5]
        x = self.pool(x)

        # [batch, 16, 5, 5] -> [batch, 120, 1, 1]
        x = self.conv3(x)
        x = self.relu(x)

        # [batch, 120, 1, 1] -> [batch, 120]
        x = x.reshape(x.size(0), 120)

        # Fully-connected layers
        x = self.fc1(x)
        x = self.relu(x)
        y = self.fc2(x)

        return y

model = MyLeNet().to(device)
print(model)


## 3. Train

In [ ]:
# CrossEntropyLoss internally applies Softmax
loss_function = nn.CrossEntropyLoss()

# Use Adam optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
)

# Training settings
num_epochs = 20
batch_size = 64

# Move data to selected device (CPU or GPU)
X_train = X_train.to(device)
y_train = y_train.to(device)
X_val = X_val.to(device)
y_val = y_val.to(device)

best_model_val_acc = 0.0
best_epoch = 0
best_model = None

for epoch in range(num_epochs):

    # Set the model to training mode
    model.train()

    total_loss = 0

    # Divide the data into mini-batches
    for start in range(0, len(X_train), batch_size):

        end = start + batch_size

        # Get one mini-batch
        images = X_train[start:end]
        labels = y_train[start:end]

        # Add channel dimension: [batch, 32, 32] -> [batch, 1, 32, 32]
        images = images.unsqueeze(1)

        # Forwarding
        outputs = model(images)

        # Compute loss
        loss = loss_function(outputs, labels)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()

        # Update weights
        optimizer.step()

        total_loss += loss.item()

    # Average training loss
    avg_loss = total_loss / ((len(X_train) + batch_size - 1) // batch_size)

    # Evaluate on the validation set
    model.eval()

    with torch.no_grad():
        X_val_cnn = X_val.unsqueeze(1)
        val_outputs = model(X_val_cnn)
        val_predictions = val_outputs.argmax(dim=1)

        val_accuracy = 100 * (val_predictions == y_val).float().mean().item()

    # Save the model with the highest validation accuracy
    if val_accuracy > best_model_val_acc:
        best_model_val_acc = val_accuracy
        best_epoch = epoch + 1
        best_model = copy.deepcopy(model)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Loss: {avg_loss:.4f} | "
        f"Val Accuracy: {val_accuracy:.2f}%"
    )


## 4. Evaluate Best Model

In [ ]:
# Use the model with the highest validation accuracy
best_model.eval()

# Move the test set to selected device (CPU or GPU)
X_test = X_test.to(device)
y_test = y_test.to(device)

# Evaluate the selected model on the test set
with torch.no_grad():
    X_test_cnn = X_test.unsqueeze(1)
    test_outputs = best_model(X_test_cnn)
    test_predictions = test_outputs.argmax(dim=1)

    test_accuracy = 100 * (test_predictions == y_test).float().mean().item()

print(f"Best Epoch: {best_epoch}")
print(f"Best Validation Accuracy: {best_model_val_acc:.2f}%")
print(f"Test Accuracy: {test_accuracy:.2f}%")


## 5. Check Learned Model

In [ ]:
# Randomly select 3 test images
indices = torch.randperm(len(X_test), device=device)[:3]

X_check = X_test[indices]
y_check = y_test[indices]

# Set the model to evaluation mode
best_model.eval()

# Disable gradient calculation
with torch.no_grad():
    X_check_cnn = X_check.unsqueeze(1)
    outputs = best_model(X_check_cnn)
    predictions = outputs.argmax(dim=1)

# Show 3 images
fig, axes = plt.subplots(1, 3, figsize=(9, 3))

for i in range(3):
    axes[i].imshow(X_check[i].cpu(), cmap="gray")
    axes[i].set_title(f"True: {y_check[i].item()} / Pred: {predictions[i].item()}")
    axes[i].axis("off")

plt.tight_layout()
plt.show()


# 6. ONNX Deployment

In [ ]:
!pip install -q onnx 
!pip install -q onnxruntime
!pip install -q onnxscript

from google.colab import files

# 1) Convert PyTorch -> ONNX (using the best model)
best_model.eval()  # Switch to inference mode

onnx_path = "my_lenet.onnx"
dummy_input = torch.randn(1, 1, 32, 32, device=device)  # Dummy input: (N, C, H, W)

torch.onnx.export(
    best_model,
    dummy_input,
    onnx_path,
    input_names=["input"],
    output_names=["logits"],
    dynamic_shapes={"x": {0: "batch"}},
    external_data=False,
)

# 2) Download the .onnx file
files.download(onnx_path)

# 7. Predicte label7.png

In [ ]:
import numpy as np
import onnxruntime as ort
from PIL import Image, ImageOps

# Load a 32x32 grayscale PNG and scale to [0, 1]
img = Image.open("label7.png").convert("L")
img = ImageOps.invert(img)

x = np.array(img, dtype=np.float32) / 255.0
x = x.reshape(1, 1, 32, 32)  # (N, C, H, W)

# Load the ONNX model and run inference
sess = ort.InferenceSession("my_lenet.onnx", providers=["CPUExecutionProvider"])
logits = sess.run(["logits"], {"input": x})

# Print the class with the largest logit
print("Predicted:", int(logits.argmax()))
